# TP 3.2 — Mini data lake « Medallion » (IVAL GT)

**Couches** : `bronze` (brut), `silver` (nettoyé), `gold` (agrégat métier).

**Source** : `donnees/fr-en-indicateurs-de-resultat-des-lycees-gt_v2.csv` — URLs dans `Docs/sources_data.md` (TP 3.2).

In [ ]:
from pathlib import Path

def resolve_data_dir() -> Path:
    """Racine du dépôt = dossier qui contient `donnees/`. Lancer Jupyter depuis cette racine."""
    cwd = Path.cwd()
    if (cwd / "donnees").is_dir():
        return cwd
    p = cwd
    for _ in range(4):
        if (p / "donnees").is_dir():
            return p
        p = p.parent
    return cwd

ROOT = resolve_data_dir()
DATA = ROOT / "donnees"
print("Répertoire données :", DATA.resolve())


In [ ]:
from pathlib import Path

BRONZE = DATA / "lake" / "bronze"
SILVER = DATA / "lake" / "silver"
GOLD = DATA / "lake" / "gold"
for p in (BRONZE, SILVER, GOLD):
    p.mkdir(parents=True, exist_ok=True)
print(BRONZE, SILVER, GOLD)


In [ ]:
import shutil
src = DATA / "fr-en-indicateurs-de-resultat-des-lycees-gt_v2.csv"
if not src.exists():
    raise FileNotFoundError(src)
dst = BRONZE / "ival_gt.csv"
shutil.copy2(src, dst)
print("Bronze :", dst.stat().st_size, "octets")


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("TP32_Lake").getOrCreate()

bronze_path = str(BRONZE / "ival_gt.csv")
df_raw = spark.read.csv(bronze_path, header=True, inferSchema=True, sep=";")

# Silver : normaliser les libellés DEPP (accents, espaces) en noms stables
rename_map = {
    "Année": "annee",
    "UAI": "uai",
    "Secteur": "secteur",
    "Académie": "libelle_academie",
    "Taux de réussite - Toutes séries": "taux_reu_total",
    "Valeur ajoutée du taux de réussite - Toutes séries": "va_reu_total",
}
df = df_raw
for src, dst in rename_map.items():
    df = df.withColumnRenamed(src, dst)

# Doublons potentiels sur clé métier
key_cols = ["annee", "uai"]
dup = df.groupBy(*key_cols).count().filter(F.col("count") > 1)
print("Doublons annee+uai :", dup.limit(5).count())

# Valeurs manquantes sur taux principal : exclure lignes sans résultat publié
clean = (
    df.dropDuplicates(key_cols)
    .withColumn("taux_reu_total", F.regexp_replace(F.col("taux_reu_total"), ",", ".").cast("double"))
    .withColumn("va_reu_total", F.regexp_replace(F.col("va_reu_total"), ",", ".").cast("double"))
    .filter(F.col("taux_reu_total").isNotNull())
)

silver_path = str(SILVER / "ival_gt_clean.parquet")
clean.write.mode("overwrite").parquet(silver_path)
print("Silver écrit :", silver_path)


In [ ]:
# Gold : moyennes par académie et secteur (agrégation métier)
silver = spark.read.parquet(silver_path)

gold = (
    silver.groupBy("annee", "libelle_academie", "secteur")
    .agg(
        F.avg("taux_reu_total").alias("taux_reu_moyen"),
        F.avg("va_reu_total").alias("va_reu_moyenne"),
        F.count("*").alias("nb_lycees"),
    )
    .orderBy("annee", "libelle_academie", "secteur")
)

gold_path = str(GOLD / "ival_par_academie_secteur")
gold.write.mode("overwrite").partitionBy("annee").parquet(gold_path)
gold.show(15, truncate=False)
print("Gold :", gold_path)


In [ ]:
from pathlib import Path

def dir_size(p: Path) -> int:
    if not p.exists():
        return 0
    if p.is_file():
        return p.stat().st_size
    return sum(dir_size(c) for c in p.iterdir())

for label, p in [("bronze", BRONZE), ("silver", SILVER), ("gold", GOLD)]:
    print(label, "octets ≈", dir_size(p))


In [ ]:
spark.stop()


## Synthèse

Comparer tailles disque et temps de lecture entre bronze (CSV), silver (Parquet typé), gold (Parquet partitionné). Noter le rôle de chaque couche pour la gouvernance et la réutilisation.